# Assignment 1 — TensorFlow Setup, Preprocessing and Visualization

**Platform:** Google Colab &nbsp;|&nbsp; **Suggested runtime:** CPU  
**How to use:** Run the cells from top to bottom. Change the small experiment
constants when more training time is available.

This workbook is written as a compact college assignment: it explains the
problem, implements the method, evaluates the result, and records the main
observations.


## Problem and objectives

Prepare a clean machine-learning dataset before modelling. The Iris
dataset is used because it is small, labelled, and easy to inspect.

By the end, we will:

1. verify TensorFlow/Keras in Colab;
2. inspect missing values and class balance;
3. split the data using stratification;
4. normalize features without leaking test information; and
5. visualize useful patterns.


In [ ]:
# Colab already includes TensorFlow. Uncomment only if an update is needed.
# %pip install -q -U tensorflow scikit-learn seaborn

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


## Load and inspect the data

Each row is a flower. The four measurements are inputs and `species`
is the class label. Initial inspection catches missing values, wrong
data types, unusual ranges, and class imbalance.


In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.rename(columns={"target": "species"})
df["species_name"] = df["species"].map(dict(enumerate(iris.target_names)))

display(df.head())
print("Shape:", df.shape)
print(); print("Missing values:"); print(df.isna().sum())
print(); print("Class counts:"); print(df["species_name"].value_counts())
display(df.describe().round(2))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x="species_name", hue="species_name", legend=False, ax=axes[0])
axes[0].set_title("Class distribution")

sns.scatterplot(
    data=df, x="petal length (cm)", y="petal width (cm)",
    hue="species_name", s=70, ax=axes[1]
)
axes[1].set_title("Petal measurements by species")
plt.tight_layout()
plt.show()

sns.heatmap(df[iris.feature_names].corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Feature correlation")
plt.show()


## Split and normalize

Stratification keeps the three classes in similar proportions. The
scaler is fitted **only on training data**, which prevents information
from the test set leaking into training.


In [ ]:
X = df[iris.feature_names].to_numpy(dtype="float32")
y = df["species"].to_numpy(dtype="int32")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype("float32")
X_test_scaled = scaler.transform(X_test).astype("float32")

print("Train/test shapes:", X_train_scaled.shape, X_test_scaled.shape)
print("Training means:", X_train_scaled.mean(axis=0).round(3))
print("Training standard deviations:", X_train_scaled.std(axis=0).round(3))
print("Test class counts:", np.bincount(y_test))


## Observations

- Iris has no missing values and its classes are balanced.
- Petal length and petal width show clear class separation.
- Standardization places training features near mean 0 and standard
  deviation 1, which usually helps gradient-based learning.
- A test set remains untouched for final evaluation.
